## this contains the implementation of various excercises questions for video 1

### common imports & global values

In [1]:
from dataclasses import dataclass,field
import numpy as np
import torch
import torch.nn.functional as F
from abc import ABC, abstractmethod
from itertools import islice

In [2]:

@dataclass
class ModelConfigs:
    """This class contains certain hyperparameter values that can be experimented upoin
    
        Class Members
            LEARNING_RATE: int - the step size taken during the optimization in gradient based approach
            REGURLARIZATION_VALUE: float - the loss factor on applied explicitly on the weights (L1/L2) regularization
            SMOOTHING_COUNT: int - make the counts array (counting approach) away from zero so the logs make more sense (away from inf probabilities)
            BOUNDARY_CHAR: str - start and end of the sequence
            SEED_VAL: int - pseudo random generator seed for reproducible samples.
    """
    LEARNING_RATE:int = field(default=-50)
    REGULARIZATION_VALUE:float = field(default=0.01)
    SMOOTHING_COUNT: int = field(default=1)
    BOUNDARY_CHAR: str = field(default=".")
    START_BOUNDARY_CHAR: str = field(default=".")
    END_BOUNDARY_CHAR: str = field(default=".")
    SEED_VAL: int = field(default=2147483647)
    NUM_EPOCHS: int = field(default= 1000)

In [3]:
@dataclass
class Vocab:
    """Vocab is a packaging class which contains important information for language modelling tasks

        Members:
            boundary_char: str - this signifies the start and end of a sequence
            vocab_letters: list[str] - this signifies the list of unique characters that occur in the data set
            stoi: dict[str, int] - this is the mapping between the the unique characters and int values
            itos: dict[int, str] -  this is the reverse of stoi
            n_unique: int - this is the number of unique characters + the boundary character
    """
    start_boundary_char: str
    end_boundary_char: str
    vocab_letters: list[str] = field(default_factory=list)
    stoi: dict[str, int] = field(default_factory=dict)
    itos: dict[int, str] = field(default_factory=dict)
    n_unique: int = field(default=0)


In [4]:
@dataclass
class NGram:
    """Packing class that contains the information related to the NGram classes

        Class Members
            data: list[tuple[str, ...]] - pairs of ngrams identified from within the sequence of strings
            n: length of the example subset (n = 2) means bigram, (n = 3) means trigram and so on.
    """
    data: list[tuple[str, ...]] = field(default_factory=list)
    n: int = field(default=2)

In [5]:
def get_names(data_path: str = '../names.txt')->list[str]:
    """Read the names file and load the contentents into memory as a list"""
    with open(data_path, 'r') as file:
        names =  file.read().splitlines()
        names = list(set(names))
    return names

In [6]:
def create_vocab(start_boundary_char: str, end_boundary_char: str, data_set: list[str])->Vocab:
  """Creates a vocab object for the provided boundary character and list of strings


    Args:
      start_boundary_char: str - the start of a sequence
      end_boundary_char : str - the end of a sequence
      data_set: list[str] - list of character sequences that are to be modelled
    
    Returns
      vocab: Vocab - this is the vocab object that contains important character level model information.
  """


  # create a list of unique occuring characters from the set
  unique_chars = sorted(list(set(''.join(data_set))))
  
  # add boundary char at index 0
  unique_chars.insert(0, start_boundary_char)

  # create the string to int mapping for the unqiue characters
  stoi = {
    s: i
    for i, s in enumerate(unique_chars)
  }

  # reverse the above mapping to have int -> string
  itos = {
    i : s
    for s, i in stoi.items()
  }


  # package all together as a object to be used in other parts of the program
  return Vocab(
    start_boundary_char = start_boundary_char,
    end_boundary_char = end_boundary_char,
    vocab_letters= unique_chars[1:],
    stoi = stoi,
    itos=itos,
    n_unique=len(unique_chars)
  )
  



In [7]:
def generate_n_grams(data_set: list[str], start_boundary_char: str, end_boundary_char: str,n: int = 2)->NGram:
    """Function to generate the n-gram set of the input names dataset by default it generates bigram examples

    Args:
        data_set: list[str] = this is a list of strings (names)
        start_boundary_char : str = this is appended at the start of the string
        end_boundary_char: str = this is the appended at the end of the string

        n: int = if 2 we generate bigrams, 3 we generate tri-grams etc.

    Returns:
        n_grams = list[tupe[str,...]] -> this is a list of tuple each tuple will atleast have two elements (the input character and the resultant character) in case of bigrams
    """

    # resultant list
    n_grams = []

    for name in data_set:
        
        # append the boundary char
        char_list = [start_boundary_char] + list(name) + [end_boundary_char]

        for i in range(len(char_list) - n + 1):
            n_grams.append(tuple(char_list[i : i + n]))
    return NGram(
        n = n,
        data=n_grams
    )


### E0. Generalized class for dry-running bigrams and trigram models using both counting and gradient based approaches.

### E1. implementation of trigram model using the count and / or gradient based approach.

**ANS E1** : the loss for trigram is only ever so sligthly better.

In [8]:
class NGramModel:
    """Common Interface for a count based NGram implementation

        Object Members
            data_set:list[str] - list of sequences to model
            vocab:Vocab - object containing information about the ngram models vocabulary
            n_gram:NGram - information about the n_gram
            counts: torch.tensor - counts of the ngram occuring in all of the sequences.
            probs: torch.tensor - normalized counts (each row sums up to 1.) probability if the next occuring character given a n-1 context
            generator: torch.Generator - a generator object to be used across all the ngram impplementations for reproducible results
    
    """
    def __init__(self, data_set: list[str], n: int, config: ModelConfigs)->None:
        self.config = config
        self.data_set = data_set
        self.vocab = create_vocab(data_set=data_set, start_boundary_char=config.START_BOUNDARY_CHAR, end_boundary_char=config.END_BOUNDARY_CHAR)
        self.n_gram = generate_n_grams(data_set= data_set, start_boundary_char=config.START_BOUNDARY_CHAR,end_boundary_char=config.END_BOUNDARY_CHAR, n = n)
        self.counts = self.create_counts_array()
        self.probs = self.counts.float()
        self.probs /= self.probs.sum(dim=-1, keepdim=True)
        self.generator = torch.Generator().manual_seed(config.SEED_VAL)
        self.W = torch.randn(size=((self.n_gram.n - 1) * self.vocab.n_unique, self.vocab.n_unique), generator=self.generator, requires_grad=True)
    
    def create_counts_array(self)->torch.Tensor:
        """Common method to generate the counts array for an n-gram model
        """
        # use laplace smoothing from the start
        counts = torch.full(size=(self.vocab.n_unique, )*self.n_gram.n, fill_value=self.config.SMOOTHING_COUNT, dtype=torch.int32)
        n = self.n_gram.n

        # traverse through all the ngrams over all the sequences
        for data in self.n_gram.data:
            # get the indices
            indices = [self.vocab.stoi[ch] for ch in islice(data, n)]
            # increment by 1
            counts[*indices] += 1
        # return the counts
        return counts
    
    
    def calculate_nll_counting_method(self)->None:
        """Common method for all ngrams to calculate the nll loss. the lower the better
        """
        # keep a running sum of the individual log probs
        log_likelihood = 0
        n = self.n_gram.n
        # iterate over all the bigrams over the data set
        for data in self.n_gram.data:
            # get the int mapping of the characters
            indices = [self.vocab.stoi[ch] for ch in islice(data, n)]
            # get the probability assigned for the bigram pair
            p = self.probs[*indices]
            # get the log of the value
            log_prob = p.log()
            # accumualate the log prob sum
            log_likelihood += log_prob.item()
        # invert the sign so we can use a loss value
        neg_log_likelihood = -log_likelihood

        # normalize to have a easily interpretable value
        neg_log_likelihood /= len(self.n_gram.data)
        # display to the user
        print(f"{neg_log_likelihood=:.4f}")
    
    
    
    def sample_names(self, sample_size: int = 5, max_length: int = 5, method="count")->None:
        """Common logic to generate sequences using the ngram counting approach
            Args:
                sample_size: int - number of sequences to be generated.
                max_length: int - number of tokens in a sequence.
                method: count | gradient - which method to use 
            Returns: None, we just print the sequences one by one.
        """
        if method.casefold() == "count":
            for _ in range(sample_size):
                out = []
                context = [self.vocab.stoi[self.vocab.start_boundary_char]] * (self.n_gram.n - 1)

                while True:
                    p = self.probs[tuple(context)]
                    next_ix = torch.multinomial(p, num_samples=1, replacement=True, generator=self.generator).item()
                    context = context[1:] + [next_ix]
                    out.append(self.vocab.itos[next_ix])
                    if next_ix == self.vocab.stoi[self.vocab.end_boundary_char]:
                        break
                    if len(out) > max_length:
                        out.append(self.vocab.end_boundary_char)
                        break
                print(''.join(out))
        elif method.casefold() == "gradient":
            for _ in range(sample_size):
                out = []
                context = [self.vocab.stoi[self.vocab.start_boundary_char]] * (self.n_gram.n - 1)

                while True:
                    xenc = F.one_hot(torch.tensor(context, dtype=torch.int64), num_classes=self.vocab.n_unique)
                    xenc = xenc.view(1, -1).float()
                    logits = xenc @ self.W
                    counts = logits.exp()
                    probs = counts / counts.sum(dim=-1, keepdim=True)
                    next_ix = torch.multinomial(probs, replacement=True, num_samples=1, generator=self.generator).item()
                    out.append(self.vocab.itos[next_ix])
                    if next_ix == self.vocab.stoi[self.vocab.end_boundary_char]:
                        break
                    if len(out) > max_length:
                        out.append(self.vocab.end_boundary_char)
                        break
                    context = context[1:] + [next_ix]
                print(''.join(out))

            


            
    def create_data_set(self)->None:
        """
        Common method to create the dataset for the gradient based algorithm

            Creates:
                self.xs: torch.tensor - that contains the context (n - 1) characters
                self.ys: torch.tensor - that contians the target character that appears after the corresponding context.
        
        """
        xs, ys = [], []
        for data in self.n_gram.data:
            context = [self.vocab.stoi[ch] for ch in data[:-1]]
            next_char = self.vocab.stoi[data[-1]]
            xs.append(context)
            ys.append(next_char)
        self.xs = torch.tensor(xs, dtype=torch.int64)
        self.ys = torch.tensor(ys, dtype=torch.int64)
    
    def train_model(self)->None:
        """
        Common method that generates sets of weights to be optimized via the gradient based algorithm
        intuition for inputs to the model is : context * num_of_unique chars, num_unique_chars
        so for bigrams it will be : (2 - 1) * 27 , 27 which denots that for the set of input puts we encode it to map to a one hot encoding of 27 classees and 27 neurons predict the probabilities of next char each neuron signifying prob for one char in vocb
        similary for trigram  it will be (3 - 1) * 27 , 27 since we input two characters worth of input this would lead to having a 54,27 weight matrix corresponding to a 2 integers one hot encoded into 27 classes.
        this method then trains for NUM_EPOCHS defined in the config class passed to the NGramModel class.
            Creates:
                self.W - weights that try to provide numbers that model the probabilites measured after normalizing the counts in counting method.
        """
        for epoch in range(self.config.NUM_EPOCHS):
            # one hot encode the information
            xenc = F.one_hot(self.xs, num_classes=self.vocab.n_unique).float()
            # this flattens out the array (batch_size, context_size, unique_char_num) -> (batch_size * context_size, n_unique)
            xenc = xenc.view(len(self.xs), -1)

            logits = xenc @ self.W

            counts = logits.exp()

            probs = counts / counts.sum(dim=1, keepdim=True)

            loss = -probs[torch.arange(len(self.xs)), self.ys].log().mean()

            if epoch % 100 == 0:
                print(f"NLL={loss.item():.4f}")
            
            
            # zero out the gradients
            self.W.grad = None

            # do backprop
            loss.backward()

            # update the weights
            self.W.data += self.config.LEARNING_RATE * self.W.grad
    

In [9]:
# get the names
names = get_names(data_path='../names.txt')
names_mixed = get_names(data_path='../names_mixed.txt')


In [10]:
config = ModelConfigs()

In [11]:
# genrerate the bigram and trigram objects
bigrams = NGramModel(data_set=names, config=config, n=2)
trigrams = NGramModel(data_set=names, config=config, n=3)

In [12]:
bigrams_mixed = NGramModel(data_set=names_mixed, config=config, n = 2)
trigrams_mixed = NGramModel(data_set=names_mixed, config=config, n = 3)

In [13]:
bigrams.sample_names(), print("*"*25), bigrams_mixed.sample_names()

jigua.
sadryr.
konini.
ddaves.
man.
*************************
jicla.
sadryr.
kanini.
dhatas.
magari.


(None, None, None)

In [14]:
trigrams.sample_names(), print("*"*25), trigrams_mixed.sample_names()

an.
lena.
jacenc.
re.
wes.
*************************
an.
lena.
jacwmc.
re.
wasukv.


(None, None, None)

In [15]:
bigrams.sample_names(method="gradient"), print("*"*25), bigrams_mixed.sample_names(method="gradient")

rpsgqb.
n.
kdknmr.
oz.
szxjbo.
*************************
ygqbmf.
kdknmr.
oz.
szxjbo.
zw.


(None, None, None)

In [16]:
trigrams.sample_names(method="gradient"), print("*"*25), trigrams_mixed.sample_names(method="gradient")

ooaanu.
mzhllv.
zziulo.
nypgbd.
oodcnm.
*************************
zpgxmz.
zon.
zziulo.
nypgbd.
oodcnm.


(None, None, None)

In [17]:
bigrams.calculate_nll_counting_method(), bigrams_mixed.calculate_nll_counting_method()

neg_log_likelihood=2.4543
neg_log_likelihood=2.4161


(None, None)

In [18]:
trigrams.calculate_nll_counting_method(), trigrams_mixed.calculate_nll_counting_method()

neg_log_likelihood=2.0999
neg_log_likelihood=2.1798


(None, None)

In [23]:
# bigrams.create_data_set()
bigrams.train_model()

NLL=2.4550
NLL=2.4549
NLL=2.4547
NLL=2.4547
NLL=2.4546
NLL=2.4545
NLL=2.4544
NLL=2.4544
NLL=2.4543
NLL=2.4543


In [24]:
# trigrams.create_data_set()
trigrams.train_model()


NLL=2.2398
NLL=2.2397
NLL=2.2395
NLL=2.2394
NLL=2.2393
NLL=2.2392
NLL=2.2391
NLL=2.2391
NLL=2.2390
NLL=2.2390


In [27]:
bigrams.sample_names(method="count"), print("*"*25), bigrams.sample_names(method="gradient")

wyurig.
man.
e.
esuce.
mo.
*************************
duie.
erie.
iabent.
abenel.
h.


(None, None, None)

In [28]:
trigrams.sample_names(method="count"), print("*"*25), trigrams.sample_names(method="gradient")

ka.
ituarn.
glse.
makaid.
na.
*************************
ahadel.
or.
ari.
aneena.
arion.


(None, None, None)